In [1]:
import jax
import jax.numpy as jnp

from shapiq.game.table import TableGame
from shapiq.players import FinitePlayers

In [2]:
coalitions = jnp.array(
    [
        [0, 0, 0],  # empty
        [1, 0, 0],  # {0}
        [0, 1, 0],  # {1}
        [1, 1, 0],  # {0, 1}
        [0, 0, 1],  # {2}
        [1, 0, 1],  # {0, 2}
        [0, 1, 1],  # {1, 2}
        [1, 1, 1],  # {0, 1, 2}
    ],
    dtype=bool,
)
g = TableGame(jnp.array([[0.0, 1.0, 2.0, 4.0, 3.0, 5.0, 6.0, 10.0]]), FinitePlayers(3))
print("game batch:", g.shape, "shared players:", g.players.shape)
for coalition, value in zip(coalitions.tolist(), g(coalitions).tolist(), strict=True):
    print(coalition, "->", value)

game batch: (1,) shared players: ()
[False, False, False] -> 0.0
[True, False, False] -> 1.0
[False, True, False] -> 2.0
[True, True, False] -> 4.0
[False, False, True] -> 3.0
[True, False, True] -> 5.0
[False, True, True] -> 6.0
[True, True, True] -> 10.0


# Build the ExactComputer for Shapley

In [3]:
from shapiq.game.exact import ExactComputer
from shapiq.index import ShapleyValue

computer = ExactComputer.from_game(g)

shapley_index = ShapleyValue()
shapley_values = computer(shapley_index)
shapley_values

AdditiveMoebiusGame(values=Array([[2.3333304, 3.333329 , 4.333328 ]], dtype=float32), players=FinitePlayers(ids=Array([0, 1, 2], dtype=int32), players=None, latent_n_players=3, n_players=3))

In [4]:
from shapiq.game.exact import ExactComputer
from shapiq.index import ShapleyInteractionIndex, ShapleyValue

computer = ExactComputer.from_game(g)

shapley_index = ShapleyInteractionIndex(max_order=2)
shapley_values = computer(shapley_index)
shapley_values

SetMoebiusGame(values=Array([[2.3333304, 3.333329 , 1.4999988, 4.333328 , 1.4999986, 1.4999988]],      dtype=float32), players=FinitePlayers(ids=Array([0, 1, 2], dtype=int32), players=None, latent_n_players=3, n_players=3), interactions=Array([[ True, False, False],
       [False,  True, False],
       [ True,  True, False],
       [False, False,  True],
       [ True, False,  True],
       [False,  True,  True]], dtype=bool), additivity=2)

# One cached table, many indices

`ExactComputer.from_game` evaluates the game once on all `2**n` coalitions and stores them as a `TableGame`. Every index computed afterwards reuses that table. The index decides which game the result is: semivalues give an `AdditiveMoebiusGame`, interaction indices a `SetMoebiusGame`.

In [5]:
from shapiq.game.weights import DDWeightedBanzhafWeight
from shapiq.index import (
    BanzhafInteractionIndex,
    BanzhafValue,
    FaithfulBanzhafTopOrderIndex,
    FaithfulShapleyTopOrderIndex,
    ShapleyInteractionIndex,
    ShapleyValue,
    WeightedBanzhafValue,
)

computer = ExactComputer.from_game(g)


def show(game):
    """Print each interaction set of the first game with its coefficient."""
    print(type(game).__name__, "batch", game.shape)
    for members, value in zip(game.interactions.tolist(), game.values[0].tolist(), strict=True):
        print("  ", {i for i, m in enumerate(members) if m}, f"{value:+.4f}")

## Semivalues: one value per player

`ShapleyValue` and `BanzhafValue` need no arguments. `WeightedBanzhafValue` takes the weight `p`, passed by keyword like every index field.

In [6]:
show(computer(ShapleyValue()))
show(computer(BanzhafValue()))
show(computer(WeightedBanzhafValue(weights=DDWeightedBanzhafWeight(0.3))))

AdditiveMoebiusGame batch (1,)
   {0} +2.3333
   {1} +3.3333
   {2} +4.3333
AdditiveMoebiusGame batch (1,)
   {0} +2.2500
   {1} +3.2500
   {2} +4.2500
AdditiveMoebiusGame batch (1,)
   {0} +1.6900
   {1} +2.6900
   {2} +3.6900


## Interaction indices: coefficients on sets up to `max_order`

SII and BII cover all sets from `min_order=1` to `max_order`. Reading these values as Möbius coefficients is an interpretation: for orders above one, the resulting game generally differs from `g`.

In [7]:
show(computer(ShapleyInteractionIndex(max_order=2)))
show(computer(BanzhafInteractionIndex(max_order=2)))

SetMoebiusGame batch (1,)
   {0} +2.3333
   {1} +3.3333
   {0, 1} +1.5000
   {2} +4.3333
   {0, 2} +1.5000
   {1, 2} +1.5000
SetMoebiusGame batch (1,)
   {0} +2.2500
   {1} +3.2500
   {0, 1} +1.5000
   {2} +4.2500
   {0, 2} +1.5000
   {1, 2} +1.5000


## Top-order Faithful indices

Top-order FSII uses the Beta(k, k) weights, top-order FBII the Banzhaf weights. Both cover only the sets of size `max_order`. For `k = 1`, FSII is the Shapley value.

In [8]:
show(computer(FaithfulShapleyTopOrderIndex(max_order=2)))
show(computer(FaithfulBanzhafTopOrderIndex(max_order=2)))

fsii_1 = computer(FaithfulShapleyTopOrderIndex(max_order=1)).values
print(
    "FSII at k = 1 equals Shapley:",
    bool(jnp.allclose(fsii_1, computer(ShapleyValue()).values, atol=1e-5)),
)

SetMoebiusGame batch (1,)
   {0, 1} +1.5000
   {0, 2} +1.5000
   {1, 2} +1.5000
SetMoebiusGame batch (1,)
   {0, 1} +1.5000
   {0, 2} +1.5000
   {1, 2} +1.5000
FSII at k = 1 equals Shapley: True


## The result is a game

The Shapley game is additive: a coalition's value is the sum of its members' Shapley values. Its grand coalition therefore gets `v(N) - v(∅)` (efficiency).

In [9]:
shapley_game = computer(ShapleyValue())
print("coalition values:", shapley_game(coalitions[:, None]).ravel().tolist())
print(
    "grand coalition:",
    float(shapley_game(jnp.ones(3, dtype=bool))[0]),
    "= v(N) - v(∅) =",
    float(g.values[0, -1] - g.values[0, 0]),
)

coalition values: [0.0, 2.3333303928375244, 3.33332896232605, 5.666659355163574, 4.333327770233154, 6.666658401489258, 7.666656494140625, 9.99998664855957]
grand coalition: 9.99998664855957 = v(N) - v(∅) = 10.0


## Batches: every game with every weight

Game and index batches broadcast like arrays. To pair every game with every `p`, give the computer a trailing axis.

In [10]:
games = TableGame(
    jnp.array(
        [[0.0, 1.0, 2.0, 4.0, 3.0, 5.0, 6.0, 10.0], [0.0, 2.0, 1.0, 3.0, 1.0, 3.0, 2.0, 4.0]]
    ),
    FinitePlayers(3),
)
batched = ExactComputer.from_game(games)
p_values = DDWeightedBanzhafWeight(jnp.array([0.3, 0.5, 0.7]))

every_pair = batched[:, None](WeightedBanzhafValue(weights=p_values))
print("games x p:", every_pair.shape)
print(every_pair.values)

games x p: (2, 3)
[[[1.69       2.69       3.69      ]
  [2.25       3.25       4.25      ]
  [2.8899999  3.8899999  4.89      ]]

 [[1.9999999  0.99999994 1.        ]
  [2.         1.         1.        ]
  [2.         1.         0.99999994]]]


## Compiling the whole computation with `jax.jit`

Games, indices and computers are pytrees, so the full pipeline, from evaluating every coalition to building the result game, compiles into one function, and the result stays a game. Each index type compiles once; afterwards, new games or new `p` values with the same shapes reuse the compiled version. A different `n_players` or `max_order` changes the shapes and compiles again.

In [11]:
@jax.jit
def compute(game, index):
    return ExactComputer.from_game(game)(index)


shapley_games = compute(games, ShapleyValue())  # compiles for ShapleyValue
print(type(shapley_games).__name__, shapley_games.shape)
print(shapley_games.values)

doubled = TableGame(2 * games.values, FinitePlayers(3))
print("doubled game:", compute(doubled, ShapleyValue()).values.tolist())  # reuses it

for p in (0.3, 0.7):  # compiles once for WeightedBanzhafValue, then reuses it
    banzhaf = compute(games, WeightedBanzhafValue(weights=DDWeightedBanzhafWeight(p)))
    print(f"p = {p}:", banzhaf.values.tolist())

AdditiveMoebiusGame (2,)
[[2.3333302 3.3333292 4.3333273]
 [1.9999971 0.9999987 0.9999986]]
doubled game: [[4.666660308837891, 6.666658401489258, 8.666654586791992], [3.9999942779541016, 1.9999973773956299, 1.9999972581863403]]
p = 0.3: [[1.6899999380111694, 2.690000057220459, 3.690000057220459], [1.9999998807907104, 1.0, 1.0]]
p = 0.7: [[2.890000104904175, 3.8899998664855957, 4.889999866485596], [2.0, 1.0, 0.9999999403953552]]
